# Contrastive Foundation

## Imports

In [1]:
import os, json, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")

PyTorch: 2.10.0+cu128
CUDA:    True
GPU:     NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM:    95.0GB


## Path & Configs

In [2]:
# Model paths 
MINILM_BASE  = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/ms-marco-MiniLM-L12-v2"
BGE_DIR      = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/bge-reranker-v2-m3"
PHORANKER    = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/PhoRanker"
STAGE_A_CKPT = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_a_no_mmarco"

# Data paths 
DOMAIN_TRAIN = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/domain_train_final_train.jsonl"
DOMAIN_DEV   = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/domain_train_final_dev.jsonl"
MMARCO       = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/reranker_data/train_triplets.jsonl"
RERANK_991   = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/retrieve_rerank_991.jsonl"
GOLD_CHUNKS  = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/gold_chunks_judged.jsonl"
MARGIN_TRAIN = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/domain_train_with_teacher_scores.jsonl" 

# Test paths 
TEST_Q       = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/question.json"
RETRIEVE_TEST = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/retrieve_test.jsonl"

# Output dirs 
CKPT_STAGE_A_WITH_MMARCO    = "/kaggle/working/ablation/stage_a_with_mmarco"
CKPT_STAGE_A_NO_MMARCO      = "/kaggle/working/ablation/stage_a_no_mmarco"
CKPT_STAGE_B_LISTWISE       = "/kaggle/working/ablation/stage_b_listwise"
CKPT_STAGE_B_MARGIN_MSE     = "/kaggle/working/ablation/stage_b_margin_mse"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("\nPaths configured ✓")

Device: cuda

Paths configured ✓


## Classes (Datasets, Loss, Eval)

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# PairwiseDataset 
class PairwiseDataset(Dataset):
    def __init__(self, paths, tokenizer, max_length=512):
        self.data = []
        for p in paths:
            with open(p) as f:
                for line in f:
                    self.data.append(json.loads(line))
        self.tok     = tokenizer
        self.max_len = max_length

    def encode(self, query, passage):
        return self.tok(
            query, passage,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        d   = self.data[idx]
        pos = self.encode(d["query"], d["positive"])
        neg = self.encode(d["query"], d["negative"])
        return {
            "pos_input_ids":      pos["input_ids"].squeeze(),
            "pos_attention_mask": pos["attention_mask"].squeeze(),
            "neg_input_ids":      neg["input_ids"].squeeze(),
            "neg_attention_mask": neg["attention_mask"].squeeze(),
        }

    def __len__(self):
        return len(self.data)


class ListwiseKDDataset(Dataset):
    def __init__(self, rerank_path, tokenizer,
                 max_length=512, max_candidates=20):
        self.tok      = tokenizer
        self.max_len  = max_length
        self.max_cand = max_candidates
        self.records  = []
        with open(rerank_path) as f:
            for line in f:
                self.records.append(json.loads(line))
        print(f"KD records: {len(self.records)}")

    def __getitem__(self, idx):
        d          = self.records[idx]
        query      = d["question"]
        candidates = d["candidates"][:self.max_cand]
        encodings, scores = [], []
        for c in candidates:
            enc = self.tok(
                query, c["chunk"],
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            encodings.append({
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
            })
            scores.append(c["bge_score"])
        return {
            "encodings":  encodings,
            "bge_scores": torch.tensor(scores, dtype=torch.float),
        }

    def __len__(self):
        return len(self.records)

# RankNet/ADR-MSE 
class ListwiseRankDataset(Dataset):
    def __init__(self, rerank_path, tokenizer,    
                 max_length=512, max_candidates=20):
        self.tok      = tokenizer
        self.max_len  = max_length
        self.max_cand = max_candidates
        self.records  = []
        with open(rerank_path) as f:
            for line in f:
                self.records.append(json.loads(line))
        print(f"Total records: {len(self.records)}")

    def __getitem__(self, idx):
        d          = self.records[idx]
        query      = d["question"]
        candidates = d["candidates"][:self.max_cand]

        encodings, bge_scores, ranks = [], [], []
        for c in candidates:
            enc = self.tok(
                query, c["chunk"],
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            encodings.append({
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
            })
            bge_scores.append(c["bge_score"])
            ranks.append(c["rank"])  # 0=best, 19=worst

        return {
            "encodings":  encodings,
            "bge_scores": torch.tensor(bge_scores, dtype=torch.float),
            "ranks":      torch.tensor(ranks,      dtype=torch.float),
        }

    def __len__(self):
        return len(self.records)

def collate_listwise_rank(batch):
    all_ids, all_masks, all_scores, all_ranks, sizes = [], [], [], [], []
    for item in batch:
        n = len(item["encodings"])
        sizes.append(n)
        for enc in item["encodings"]:
            all_ids.append(enc["input_ids"])
            all_masks.append(enc["attention_mask"])
        all_scores.append(item["bge_scores"])
        all_ranks.append(item["ranks"])
    return {
        "input_ids":      torch.stack(all_ids),
        "attention_mask": torch.stack(all_masks),
        "bge_scores":     all_scores,   # list of tensors
        "ranks":          all_ranks,    # list of tensors
        "sizes":          sizes,
    }


# ADR-MSE Loss 
class ADRMSELoss(nn.Module):
    """Asymmetric Distillation Ranking MSE
    Student margin matches teacher RANK margin (normalized)
    rank: 0=best → normalize về [-1, +1]
    """
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, student_logits, teacher_ranks):
        # teacher_ranks: [0, 1, 2, ..., 19] → normalize về [-1, +1]
        # rank 0 (best) → +1, rank 19 (worst) → -1
        n = teacher_ranks.size(-1)
        rank_signal = 1.0 - 2.0 * teacher_ranks / (n - 1)  # [+1, ..., -1]

        # Normalize student logits về cùng scale
        s_mean = student_logits.mean(dim=-1, keepdim=True)
        s_std  = student_logits.std(dim=-1, keepdim=True) + 1e-8
        student_norm = (student_logits - s_mean) / s_std

        return self.mse(student_norm, rank_signal)


# RankNet Loss với teacher soft labels 
class RankNetSoftLoss(nn.Module):
    """RankNet (Burges et al., 2005) với teacher soft labels từ BGE-M3
    Thay vì binary label, dùng P_ij = σ(T_i - T_j) làm target
    """
    def __init__(self):
        super().__init__()

    def forward(self, student_logits, teacher_scores):
        n = student_logits.size(-1)

        # Tính teacher prob P_ij = σ(T_i - T_j) cho mọi cặp (i, j)
        # student_logits: [batch, n]
        s_diff = student_logits.unsqueeze(2) - student_logits.unsqueeze(1)  # [B, n, n]
        t_diff = teacher_scores.unsqueeze(2) - teacher_scores.unsqueeze(1)  # [B, n, n]

        teacher_prob = torch.sigmoid(t_diff)  # P_ij từ teacher
        student_prob = torch.sigmoid(s_diff)  # Predicted P_ij từ student

        # BCE loss trên tất cả các cặp, loại diagonal (i==j)
        mask = ~torch.eye(n, dtype=torch.bool, device=student_logits.device)
        loss = F.binary_cross_entropy(
            student_prob[:, mask],
            teacher_prob[:, mask],
        )
        return loss

def collate_listwise(batch):
    all_ids, all_masks, all_scores, sizes = [], [], [], []
    for item in batch:
        sizes.append(len(item["encodings"]))
        for enc in item["encodings"]:
            all_ids.append(enc["input_ids"])
            all_masks.append(enc["attention_mask"])
        all_scores.append(item["bge_scores"])
    return {
        "input_ids":      torch.stack(all_ids),
        "attention_mask": torch.stack(all_masks),
        "bge_scores":     all_scores,
        "sizes":          sizes,
    }


# MarginMSEDataset (Stage B Margin-MSE) 
class MarginMSEDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=512):
        self.data = []
        with open(path) as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tok     = tokenizer
        self.max_len = max_length

    def encode(self, query, passage):
        return self.tok(
            query, passage,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        d   = self.data[idx]
        pos = self.encode(d["query"], d["positive"])
        neg = self.encode(d["query"], d["negative"])
        return {
            "pos_input_ids":      pos["input_ids"].squeeze(),
            "pos_attention_mask": pos["attention_mask"].squeeze(),
            "neg_input_ids":      neg["input_ids"].squeeze(),
            "neg_attention_mask": neg["attention_mask"].squeeze(),
            "teacher_margin":     torch.tensor(d["teacher_margin"], dtype=torch.float),
        }

    def __len__(self):
        return len(self.data)


# Loss functions
class StageALoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pos_logits, neg_logits):
        return (self.bce(pos_logits, torch.ones_like(pos_logits)) +
                self.bce(neg_logits, torch.zeros_like(neg_logits))) / 2


class ListwiseKLLoss(nn.Module):
    def __init__(self, temperature=2.0):
        super().__init__()
        self.T = temperature

    def forward(self, student_logits, teacher_scores):
        s = F.log_softmax(student_logits / self.T, dim=-1)
        t = F.softmax(teacher_scores    / self.T, dim=-1)
        return F.kl_div(s, t, reduction="batchmean") * (self.T ** 2)


class MarginMSELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, pos_logits, neg_logits, teacher_margin):
        return self.mse(pos_logits - neg_logits, teacher_margin)


# Dev evaluation (pairwise accuracy)
def evaluate_pairwise(model, dev_loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in dev_loader:
            pos = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device),
            ).logits
            neg = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device),
            ).logits
            correct += (pos > neg).sum().item()
            total   += pos.size(0)
    return correct / total


print("Cell 03 done — classes & losses loaded ✓")

Cell 03 done — classes & losses loaded ✓


In [4]:
class DiscountedADRMSELoss(nn.Module):
    """ADR-MSE với nDCG-style discount weight
    Top positions get higher gradient → trực tiếp tối ưu MRR/NDCG
    """
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss(reduction='none')

    def forward(self, student_logits, teacher_ranks):
        n = teacher_ranks.size(-1)
        
        # Normalize ranks về [-1, +1]
        rank_signal = 1.0 - 2.0 * teacher_ranks / (n - 1)
        
        # Normalize student logits
        s_mean = student_logits.mean(dim=-1, keepdim=True)
        s_std  = student_logits.std(dim=-1, keepdim=True) + 1e-8
        student_norm = (student_logits - s_mean) / s_std

        # nDCG-style discount: w_i = 1/log2(rank_i + 2)
        # rank 0 → w=1.0, rank 1 → w=0.63, rank 19 → w=0.23
        weights = 1.0 / torch.log2(teacher_ranks + 2.0)
        weights = weights / weights.sum(dim=-1, keepdim=True)  # normalize

        elementwise = self.mse(student_norm, rank_signal)
        return (weights * elementwise).sum(dim=-1).mean()

In [11]:
class TopKADRMSELoss(nn.Module):
    """Chỉ tính loss trên top-k positions của teacher ranking
    Bỏ noise từ tail, focus gradient vào những gì quan trọng
    """
    def __init__(self, k=10):
        super().__init__()
        self.k   = k
        self.mse = nn.MSELoss()

    def forward(self, student_logits, teacher_ranks):
        n = teacher_ranks.size(-1)
        
        # Chỉ lấy top-k (rank 0, 1, ..., k-1)
        topk_mask = (teacher_ranks < self.k)  # (batch, n)
        
        rank_signal = 1.0 - 2.0 * teacher_ranks / (n - 1)
        s_mean = student_logits.mean(dim=-1, keepdim=True)
        s_std  = student_logits.std(dim=-1, keepdim=True) + 1e-8
        student_norm = (student_logits - s_mean) / s_std

        # Chỉ tính loss trên top-k candidates
        s_topk = student_norm[topk_mask]
        r_topk = rank_signal[topk_mask]
        
        return self.mse(s_topk, r_topk)

## Train stage A

In [4]:
def train_stage_a(
    domain_train_path,
    mmarco_path,          # None = không dùng mMARCO
    dev_path,
    output_dir,
    base_model=None,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    max_length=512,
    domain_upsample=8,
    patience=2,
    seed=42,
):
    if base_model is None:
        base_model = MINILM_BASE

    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model     = AutoModelForSequenceClassification.from_pretrained(
        base_model, num_labels=1, ignore_mismatched_sizes=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Base model: {base_model.split('/')[-1]}")
    print(f"Device: {device} | Seed: {seed}")

    # Build training paths
    domain_paths = [domain_train_path] * domain_upsample
    if mmarco_path:
        all_paths = domain_paths + [mmarco_path]
    else:
        all_paths = domain_paths

    train_dataset = PairwiseDataset(all_paths, tokenizer, max_length)
    dev_dataset   = PairwiseDataset([dev_path], tokenizer, max_length)

    domain_n = 2689 * domain_upsample
    mmarco_n = len(train_dataset) - domain_n
    print(f"Train: {len(train_dataset):,} (domain {domain_n:,} | mmarco ~{mmarco_n:,})")
    print(f"Dev:   {len(dev_dataset):,}")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=0,
        worker_init_fn=lambda w: set_seed(seed + w),
    )
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )
    criterion  = StageALoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc   = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            pos = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device),
            ).logits
            neg = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device),
            ).logits
            loss = criterion(pos, neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n   = len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | Dev Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tokenizer.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nStage A done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"


print("Cell 04 done — train_stage_a loaded ✓")

Cell 04 done — train_stage_a loaded ✓


## Train stage B

In [5]:
def train_stage_b(
    stage_a_checkpoint,
    rerank_path,
    domain_train_path,
    dev_path,
    output_dir,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=512,
    temperature=2.0,
    alpha=0.7,
    patience=2,
    seed=42,
):
    set_seed(seed)
    reranker_tok = AutoTokenizer.from_pretrained(stage_a_checkpoint)
    model        = AutoModelForSequenceClassification.from_pretrained(stage_a_checkpoint)
    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Checkpoint: {stage_a_checkpoint.split('/')[-2]}/{stage_a_checkpoint.split('/')[-1]}")
    print(f"Device: {device} | Seed: {seed} | alpha={alpha} | T={temperature}")

    kd_dataset = ListwiseKDDataset(rerank_path, reranker_tok, max_length)
    kd_loader  = DataLoader(
        kd_dataset, batch_size=batch_size,
        shuffle=True, collate_fn=collate_listwise,
        num_workers=0, worker_init_fn=lambda w: set_seed(seed + w),
    )

    cl_dataset = PairwiseDataset([domain_train_path], reranker_tok, max_length)
    cl_loader  = DataLoader(cl_dataset, batch_size=batch_size * 2, shuffle=True, num_workers=0)
    cl_iter    = iter(cl_loader)

    dev_dataset = PairwiseDataset([dev_path], reranker_tok, max_length)
    dev_loader  = DataLoader(dev_dataset, batch_size=32, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(kd_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.05 * total_steps),
        num_training_steps=total_steps,
    )
    kd_crit = ListwiseKLLoss(temperature=temperature)
    cl_crit = StageALoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc   = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = total_kd = total_cl = 0

        for batch in kd_loader:
            optimizer.zero_grad()
            all_logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            ).logits.squeeze(-1)

            kd_loss = torch.tensor(0.0, device=device)
            offset  = 0
            for i, size in enumerate(batch["sizes"]):
                kd_loss += kd_crit(
                    all_logits[offset:offset+size].unsqueeze(0),
                    batch["bge_scores"][i].to(device).unsqueeze(0),
                )
                offset += size
            kd_loss /= len(batch["sizes"])

            try:
                cl_batch = next(cl_iter)
            except StopIteration:
                cl_iter  = iter(cl_loader)
                cl_batch = next(cl_iter)

            pos = model(
                input_ids=cl_batch["pos_input_ids"].to(device),
                attention_mask=cl_batch["pos_attention_mask"].to(device),
            ).logits
            neg = model(
                input_ids=cl_batch["neg_input_ids"].to(device),
                attention_mask=cl_batch["neg_attention_mask"].to(device),
            ).logits
            cl_loss = cl_crit(pos, neg)

            loss = alpha * kd_loss + (1 - alpha) * cl_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            total_kd   += kd_loss.item()
            total_cl   += cl_loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n   = len(kd_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | "
              f"KD: {total_kd/n:.4f} | CL: {total_cl/n:.4f} | Dev Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            reranker_tok.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nStage B done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"


print("Cell 05 done — train_stage_b loaded ✓")

Cell 05 done — train_stage_b loaded ✓


## Train MarginMSE 

In [ ]:
def train_margin_mse(
    base_checkpoint,
    train_path,
    dev_path,
    output_dir,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=512,
    patience=2,
    seed=42,
):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
    model     = AutoModelForSequenceClassification.from_pretrained(
        base_checkpoint, num_labels=1, ignore_mismatched_sizes=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Checkpoint: {base_checkpoint.split('/')[-2]}/{base_checkpoint.split('/')[-1]}")
    print(f"Device: {device} | Seed: {seed}")

    train_dataset = MarginMSEDataset(train_path, tokenizer, max_length)
    dev_dataset   = PairwiseDataset([dev_path], tokenizer, max_length)
    print(f"Train: {len(train_dataset):,} | Dev: {len(dev_dataset):,}")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=0,
        worker_init_fn=lambda w: set_seed(seed + w),
    )
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )
    criterion  = MarginMSELoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc   = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            pos = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device),
            ).logits.squeeze(-1)
            neg = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device),
            ).logits.squeeze(-1)
            loss = criterion(pos, neg, batch["teacher_margin"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n   = len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | Dev Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tokenizer.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nMargin-MSE done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"


print("Cell 06 done — train_margin_mse loaded ✓")

## Train RankNet/ADR-MSE

In [12]:
def train_stage_b_ranknet(
    stage_a_checkpoint,
    rerank_path,
    domain_train_path,
    dev_path,
    output_dir,
    loss_type="ranknet",    # "ranknet" hoặc "adr_mse"
    epochs=5,
    batch_size=8,           # batch nhỏ hơn vì listwise nặng hơn
    lr=1e-5,
    max_length=512,
    alpha=0.7,              # KD weight
    patience=2,
    seed=42,
):
    set_seed(seed)
    tok   = AutoTokenizer.from_pretrained(stage_a_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(stage_a_checkpoint)
    device = torch.device("cuda")
    model.to(device)
    print(f"Loss: {loss_type} | alpha={alpha} | Seed: {seed}")

    kd_dataset = ListwiseRankDataset(rerank_path, tok, max_length)
    kd_loader  = DataLoader(
        kd_dataset, batch_size=batch_size,
        shuffle=True, collate_fn=collate_listwise_rank,
        num_workers=0, worker_init_fn=lambda w: set_seed(seed + w),
    )

    cl_dataset = PairwiseDataset([domain_train_path], tok, max_length)
    cl_loader  = DataLoader(cl_dataset, batch_size=batch_size*2, shuffle=True, num_workers=0)
    cl_iter    = iter(cl_loader)

    dev_dataset = PairwiseDataset([dev_path], tok, max_length)
    dev_loader  = DataLoader(dev_dataset, batch_size=32, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(kd_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.05 * total_steps),
        num_training_steps=total_steps,
    )

    # Chọn loss function
    if loss_type == "ranknet":
        kd_crit = RankNetSoftLoss()
    elif loss_type == "adr_mse":
        kd_crit = ADRMSELoss()
    elif loss_type == "discount":
        kd_crit = DiscountedADRMSELoss()
    elif loss_type == "topk":
        kd_crit = TopKADRMSELoss()
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

    cl_crit = StageALoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = total_kd = total_cl = 0

        for batch in kd_loader:
            optimizer.zero_grad()

            all_logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            ).logits.squeeze(-1)

            kd_loss = torch.tensor(0.0, device=device)
            offset  = 0
            for i, size in enumerate(batch["sizes"]):
                q_logits = all_logits[offset:offset+size].unsqueeze(0)

                if loss_type == "ranknet":
                    q_scores = batch["bge_scores"][i].to(device).unsqueeze(0)
                    kd_loss += kd_crit(q_logits, q_scores)
                else:  # adr_mse
                    q_ranks  = batch["ranks"][i].to(device).unsqueeze(0)
                    kd_loss += kd_crit(q_logits, q_ranks)

                offset += size
            kd_loss /= len(batch["sizes"])

            try:
                cl_batch = next(cl_iter)
            except StopIteration:
                cl_iter  = iter(cl_loader)
                cl_batch = next(cl_iter)

            pos = model(
                input_ids=cl_batch["pos_input_ids"].to(device),
                attention_mask=cl_batch["pos_attention_mask"].to(device),
            ).logits
            neg = model(
                input_ids=cl_batch["neg_input_ids"].to(device),
                attention_mask=cl_batch["neg_attention_mask"].to(device),
            ).logits
            cl_loss = cl_crit(pos, neg)

            loss = alpha * kd_loss + (1 - alpha) * cl_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            total_kd   += kd_loss.item()
            total_cl   += cl_loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n   = len(kd_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | "
              f"KD({loss_type}): {total_kd/n:.4f} | CL: {total_cl/n:.4f} | Dev Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tok.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\n{loss_type} done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"

## Benchmark Reranker

In [8]:
def benchmark_reranker(
    model,
    tokenizer,
    retrieve_results,
    questions,
    device,
    max_questions=None,
    batch_size=32,
    max_len=512,
    warmup=10,
    label="",
):
    model.eval()
    samples = retrieve_results if max_questions is None else retrieve_results[:max_questions]

    # Warmup
    print(f"\n[{label}] Warming up...")
    for entry in samples[:warmup]:
        pairs = [(entry["question"], c["chunk"]) for c in entry["candidates"]]
        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True, truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                _ = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits
    if device.type == "cuda":
        torch.cuda.synchronize()

    # Evaluate
    print(f"[{label}] Evaluating...")
    latencies, recall5, mrr10, total, total_pairs, misses = [], 0, 0, 0, 0, []

    for entry in samples:
        query = entry["question"]
        if query not in questions:
            continue

        candidates  = entry["candidates"]
        gold_ids    = set(questions[query]["gold_chunk_ids"])
        pairs       = [(query, c["chunk"]) for c in candidates]
        total_pairs += len(pairs)
        scores      = []

        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True, truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                logits = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits.squeeze(-1)
            out = logits.detach().cpu().tolist()
            scores.extend([out] if isinstance(out, float) else out)

        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)

        ranked = [c for _, c in sorted(zip(scores, candidates), key=lambda x: -x[0])]
        top5   = [c["chunk_id"] for c in ranked[:5]]

        if any(g in top5 for g in gold_ids):
            recall5 += 1
        else:
            misses.append(query)

        for rank, c in enumerate(ranked[:10], 1):
            if c["chunk_id"] in gold_ids:
                mrr10 += 1.0 / rank
                break

        total += 1

    r5  = recall5 / total
    m10 = mrr10   / total
    lat = np.mean(latencies)
    p50 = np.percentile(latencies, 50)
    p95 = np.percentile(latencies, 95)

    print(f"\n[{label}] RESULTS")
    print("=" * 60)
    print(f"Questions : {total} | Pairs: {total_pairs} | Misses: {len(misses)}")
    print("-" * 60)
    print(f"Recall@5  : {r5:.4f}")
    print(f"MRR@10    : {m10:.4f}")
    print("-" * 60)
    print(f"Latency   : avg={lat*1000:.1f}ms | p50={p50*1000:.1f}ms | p95={p95*1000:.1f}ms")
    print(f"QPS       : {total/sum(latencies):.1f}")

    return {
        "Recall@5": r5, "MRR@10": m10,
        "misses": len(misses), "total": total,
        "avg_latency_ms": lat * 1000,
        "p50_ms": p50 * 1000,
        "p95_ms": p95 * 1000,
        "qps":    total / sum(latencies),
    }


print("Cell 07 done — benchmark_reranker loaded ✓")

Cell 07 done — benchmark_reranker loaded ✓


## Load test data

In [9]:
# Load questions
with open(TEST_Q, encoding="utf-8") as f:
    raw = json.load(f)
questions = {q["question"]: q for q in raw}

# Load retrieve results
retrieve_results = []
with open(RETRIEVE_TEST, encoding="utf-8") as f:
    for line in f:
        retrieve_results.append(json.loads(line))

matched = sum(1 for e in retrieve_results if e["question"] in questions)
print(f"Questions : {len(questions)}")
print(f"Retrieve  : {len(retrieve_results)}")
print(f"Matched   : {matched}")

# Init results dict
ablation_results = {}
print("\nTest data loaded ✓")

Questions : 343
Retrieve  : 343
Matched   : 343

Test data loaded ✓


## MiniLM Base

In [ ]:
tok_base   = AutoTokenizer.from_pretrained(MINILM_BASE)
model_base = AutoModelForSequenceClassification.from_pretrained(MINILM_BASE).to(device).eval()
n_params   = round(sum(p.numel() for p in model_base.parameters()) / 1e6, 1)
print(f"MiniLM base: {n_params}M params")

ablation_results["(1) MiniLM-L12 base"] = benchmark_reranker(
    model=model_base, tokenizer=tok_base,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="MiniLM base",
)
ablation_results["(1) MiniLM-L12 base"]["params_M"] = n_params

del model_base, tok_base
torch.cuda.empty_cache()

## Stage A only with ms-macro & data domain

In [ ]:
ckpt_a_with_mmarco = train_stage_a(
    domain_train_path=DOMAIN_TRAIN,
    mmarco_path=MMARCO,
    dev_path=DOMAIN_DEV,
    output_dir=CKPT_STAGE_A_WITH_MMARCO,
    domain_upsample=8,
    patience=2,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    seed=42,
)

tok_a   = AutoTokenizer.from_pretrained(ckpt_a_with_mmarco)
model_a = AutoModelForSequenceClassification.from_pretrained(ckpt_a_with_mmarco).to(device).eval()

ablation_results["(2) Stage A only (w/ mMARCO)"] = benchmark_reranker(
    model=model_a, tokenizer=tok_a,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="Stage A w/ mMARCO",
)
ablation_results["(2) Stage A only (w/ mMARCO)"]["params_M"] = 33.4

del model_a, tok_a
torch.cuda.empty_cache()

## Stage A only with ms-marco

In [8]:
ckpt_a1 = train_stage_a(
    domain_train_path=MMARCO,
    mmarco_path=None,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_a1_mmarco",
    base_model=MINILM_BASE,
    domain_upsample=1,      
    epochs=2,               
    batch_size=32, lr=2e-5,
    patience=2, seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/tranquanghuy2809/tqhtqh/ms-marco-MiniLM-L12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Base model: ms-marco-MiniLM-L12-v2
Device: cuda | Seed: 42
Train: 50,000 (domain 2,689 | mmarco ~47,311)
Dev:   305
Epoch 1/2 | Loss: 0.4654 | Dev Acc: 0.6852


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.6852)
Epoch 2/2 | Loss: 0.3719 | Dev Acc: 0.6820
  No improve (1/2)

Stage A done. Best dev acc: 0.6852


## Next: Stage A2 (CL trên data domain)

In [9]:
ckpt_a2 = train_stage_a(
    domain_train_path=DOMAIN_TRAIN,
    mmarco_path=None,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_a2_domain",
    base_model=ckpt_a1,     
    domain_upsample=1,
    epochs=5,
    batch_size=32, lr=2e-5,
    patience=2, seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base model: best
Device: cuda | Seed: 42
Train: 2,689 (domain 2,689 | mmarco ~0)
Dev:   305
Epoch 1/5 | Loss: 0.5614 | Dev Acc: 0.8557


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8557)
Epoch 2/5 | Loss: 0.3731 | Dev Acc: 0.8590


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8590)
Epoch 3/5 | Loss: 0.3010 | Dev Acc: 0.8557
  No improve (1/2)
Epoch 4/5 | Loss: 0.2626 | Dev Acc: 0.8426
  No improve (2/2)
Early stopping at epoch 4

Stage A done. Best dev acc: 0.8590


In [10]:
tok_a2   = AutoTokenizer.from_pretrained(ckpt_a2)
model_a2 = AutoModelForSequenceClassification.from_pretrained(ckpt_a2).to(device).eval()
ablation_results["3-Stage A2 only"] = benchmark_reranker(
    model=model_a2, tokenizer=tok_a2,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="3-Stage A2 only",
)
ablation_results["3-Stage A2 only"]["params_M"] = 33.4
del model_a2, tok_a2
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[3-Stage A2 only] Warming up...
[3-Stage A2 only] Evaluating...

[3-Stage A2 only] RESULTS
Questions : 343 | Pairs: 6860 | Misses: 39
------------------------------------------------------------
Recall@5  : 0.8863
MRR@10    : 0.7293
------------------------------------------------------------
Latency   : avg=22.6ms | p50=22.6ms | p95=23.6ms
QPS       : 44.3


In [15]:
ckpt_b_3stage = train_stage_b_ranknet(
    stage_a_checkpoint=ckpt_a2,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_3stage_adrmse",
    loss_type="adr_mse",
    epochs=5, batch_size=8, lr=1e-5, alpha=0.7,
    patience=2, seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: adr_mse | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.6286 | KD(adr_mse): 0.7588 | CL: 0.3249 | Dev Acc: 0.8656


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8656)
Epoch 2/5 | Loss: 0.5827 | KD(adr_mse): 0.6925 | CL: 0.3266 | Dev Acc: 0.8754


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8754)
Epoch 3/5 | Loss: 0.5451 | KD(adr_mse): 0.6616 | CL: 0.2733 | Dev Acc: 0.8852


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8852)
Epoch 4/5 | Loss: 0.5324 | KD(adr_mse): 0.6415 | CL: 0.2778 | Dev Acc: 0.8885


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8885)
Epoch 5/5 | Loss: 0.5268 | KD(adr_mse): 0.6369 | CL: 0.2702 | Dev Acc: 0.8852
  No improve (1/2)

adr_mse done. Best dev acc: 0.8885


In [16]:
tok_3s   = AutoTokenizer.from_pretrained(ckpt_b_3stage)
model_3s = AutoModelForSequenceClassification.from_pretrained(ckpt_b_3stage).to(device).eval()
ablation_results["3-Stage A+B ADR-MSE"] = benchmark_reranker(
    model=model_3s, tokenizer=tok_3s,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="3-Stage A+B ADR-MSE",
)
ablation_results["3-Stage A+B ADR-MSE"]["params_M"] = 33.4
del model_3s, tok_3s
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[3-Stage A+B ADR-MSE] Warming up...
[3-Stage A+B ADR-MSE] Evaluating...

[3-Stage A+B ADR-MSE] RESULTS
Questions : 343 | Pairs: 6860 | Misses: 30
------------------------------------------------------------
Recall@5  : 0.9125
MRR@10    : 0.7729
------------------------------------------------------------
Latency   : avg=22.4ms | p50=22.4ms | p95=23.2ms
QPS       : 44.6


In [20]:
# ── Eval extended metrics cho 3-stage ─────────────────────────────
NEW_MODELS = {
    "3-Stage A2 only":       ("/kaggle/working/ablation/stage_a2_domain/best",        512),
    "3-Stage A+B ADR-MSE":   ("/kaggle/working/ablation/stage_b_3stage_adrmse/best",  512),
}

for name, (ckpt, max_len) in NEW_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    tok   = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()
    n_params = round(sum(p.numel() for p in model.parameters()) / 1e6, 1)

    result = compute_extended_metrics(
        model=model, tokenizer=tok,
        retrieve_results=retrieve_results, questions=questions,
        device=device, batch_size=32, max_len=max_len,
        label=name,
    )
    result["params_M"] = n_params
    extended_results[name] = result

    print(f"  Hit@1    = {result['Hit@1']:.4f}")
    print(f"  Recall@5 = {result['Recall@5']:.4f}")
    print(f"  MRR@10   = {result['MRR@10']:.4f}")
    print(f"  NDCG@10  = {result['NDCG@10']:.4f}")

    del model, tok
    torch.cuda.empty_cache()

# ── So sánh final ─────────────────────────────────────────────────
print(f"\n{'='*90}")
print(f"{'Model':<38} {'Hit@1':>7} {'Recall@5':>10} {'MRR@10':>10} {'NDCG@10':>10}")
print("-"*90)
for name in [
    "3-Stage A2 only",
    "3-Stage A+B ADR-MSE",
]:
    r = extended_results.get(name) or ablation_results.get(name)
    if not r:
        continue
    print(
        f"{name:<38} "
        f"{r.get('Hit@1',0):>7.4f} "
        f"{r.get('Recall@5',0):>10.4f} "
        f"{r.get('MRR@10',0):>10.4f} "
        f"{r.get('NDCG@10',0):>10.4f}"
    )
print("="*90)


Evaluating: 3-Stage A2 only


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Hit@1    = 0.6152
  Recall@5 = 0.8863
  MRR@10   = 0.7293
  NDCG@10  = 0.7781

Evaluating: 3-Stage A+B ADR-MSE


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Hit@1    = 0.6735
  Recall@5 = 0.9125
  MRR@10   = 0.7729
  NDCG@10  = 0.8153

Model                                    Hit@1   Recall@5     MRR@10    NDCG@10
------------------------------------------------------------------------------------------
3-Stage A2 only                         0.6152     0.8863     0.7293     0.7781
3-Stage A+B ADR-MSE                     0.6735     0.9125     0.7729     0.8153


## Stage A only (no ms-macro)

In [ ]:
ckpt_a_no_mmarco = train_stage_a(
    domain_train_path=DOMAIN_TRAIN,
    mmarco_path=None,          
    dev_path=DOMAIN_DEV,
    output_dir=CKPT_STAGE_A_NO_MMARCO,
    domain_upsample=1,        
    patience=2,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    seed=42,
)

tok_a2   = AutoTokenizer.from_pretrained(ckpt_a_no_mmarco)
model_a2 = AutoModelForSequenceClassification.from_pretrained(ckpt_a_no_mmarco).to(device).eval()

ablation_results["(3) Stage A only (no mMARCO)"] = benchmark_reranker(
    model=model_a2, tokenizer=tok_a2,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="Stage A no mMARCO",
)
ablation_results["(3) Stage A only (no mMARCO)"]["params_M"] = 33.4

del model_a2, tok_a2
torch.cuda.empty_cache()

## Stage A + Stage B (ListwiseKL)

In [ ]:
ckpt_b_kl = train_stage_b(
    stage_a_checkpoint=ckpt_a_with_mmarco,
    rerank_path=RERANK_991,
    gold_path=GOLD_CHUNKS,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=CKPT_STAGE_B_LISTWISE,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    temperature=2.0,
    alpha=0.7,
    patience=2,
    seed=42,
)

tok_b   = AutoTokenizer.from_pretrained(ckpt_b_kl)
model_b = AutoModelForSequenceClassification.from_pretrained(ckpt_b_kl).to(device).eval()

ablation_results["(4) Stage A+B (ListwiseKL)"] = benchmark_reranker(
    model=model_b, tokenizer=tok_b,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="Stage A+B ListwiseKL",
)
ablation_results["(4) Stage A+B (ListwiseKL)"]["params_M"] = 33.4

del model_b, tok_b
torch.cuda.empty_cache()

## Stage A + Stage B (MarginMSE)

In [ ]:
ckpt_b_mse = train_margin_mse(
    base_checkpoint=ckpt_a_with_mmarco,
    train_path=MARGIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=CKPT_STAGE_B_MARGIN_MSE,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    patience=2,
    seed=42,
)

tok_mse   = AutoTokenizer.from_pretrained(ckpt_b_mse)
model_mse = AutoModelForSequenceClassification.from_pretrained(ckpt_b_mse).to(device).eval()

ablation_results["(5) Stage A+B (Margin-MSE)"] = benchmark_reranker(
    model=model_mse, tokenizer=tok_mse,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="Stage A+B Margin-MSE",
)
ablation_results["(5) Stage A+B (Margin-MSE)"]["params_M"] = 33.4

del model_mse, tok_mse
torch.cuda.empty_cache()

## Stage A no macro + Stage B (ListwiseKL)

In [ ]:
ckpt_b_kl_nommarco = train_stage_b(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_kl_no_mmarco_991",
    epochs=5, batch_size=16, lr=1e-5,
    temperature=2.0, alpha=0.7,
    patience=2, seed=42,
)

tok_b2   = AutoTokenizer.from_pretrained(ckpt_b_kl_nommarco)
model_b2 = AutoModelForSequenceClassification.from_pretrained(
    ckpt_b_kl_nommarco).to(device).eval()

ablation_results["(6) No mMARCO + Stage B (KL)"] = benchmark_reranker(
    model=model_b2, tokenizer=tok_b2,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="No mMARCO + Stage B KL",
)
ablation_results["(6) No mMARCO + Stage B (KL)"]["params_M"] = 33.4

del model_b2, tok_b2
torch.cuda.empty_cache()

## Stage A no macro + Stage B (MarginMSE)

In [ ]:
ckpt_b_mse_nommarco = train_margin_mse(
    base_checkpoint=ckpt_a_no_mmarco,
    train_path=MARGIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_mse_no_mmarco",
    epochs=5, batch_size=16, lr=1e-5,
    patience=2, seed=42,
)

tok_b3   = AutoTokenizer.from_pretrained(ckpt_b_mse_nommarco)
model_b3 = AutoModelForSequenceClassification.from_pretrained(
    ckpt_b_mse_nommarco).to(device).eval()

ablation_results["(7) No mMARCO + Stage B (MSE)"] = benchmark_reranker(
    model=model_b3, tokenizer=tok_b3,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="No mMARCO + Stage B MSE",
)
ablation_results["(7) No mMARCO + Stage B (MSE)"]["params_M"] = 33.4

del model_b3, tok_b3
torch.cuda.empty_cache()

## KD thẳng từ base model - không dùng ms-macro

In [ ]:
ckpt_kd_direct = train_stage_b(
    stage_a_checkpoint=MINILM_BASE,
    rerank_path=RERANK_991,
    gold_path=GOLD_CHUNKS,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/kd_direct_from_base",
    epochs=5, batch_size=16, lr=1e-5,
    temperature=2.0, alpha=1.0,
    patience=2, seed=42,
)

tok_kd   = AutoTokenizer.from_pretrained(ckpt_kd_direct)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_kd_direct).to(device).eval()

ablation_results["(8) KD direct (no Stage A)"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="KD direct from base",
)
ablation_results["(8) KD direct (no Stage A)"]["params_M"] = 33.4
del model_kd, tok_kd
torch.cuda.empty_cache()

## KD từ Stage A checkpoint (no ms-macro)

In [ ]:
ckpt_kd_direct = train_stage_b(
    stage_a_checkpoint="/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_a_no_mmarco",
    rerank_path=RERANK_991,
    gold_path=GOLD_CHUNKS,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/kd_direct_from_base",
    epochs=5, batch_size=16, lr=1e-5,
    temperature=2.0, alpha=1.0,
    patience=2, seed=42,
)

tok_kd   = AutoTokenizer.from_pretrained(ckpt_kd_direct)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_kd_direct).to(device).eval()

ablation_results["(8) KD direct (no Stage A)"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="KD direct from base",
)
ablation_results["(8) KD direct (no Stage A)"]["params_M"] = 33.4
del model_kd, tok_kd
torch.cuda.empty_cache()

## Stage A no macro + Stage B (RankNet)

In [ ]:
ckpt_ranknet = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN, 
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_ranknet",
    loss_type="ranknet",
    epochs=5, batch_size=8, lr=1e-5, alpha=0.7, patience=2, seed=42,
)

In [ ]:
ckpt_ranknet = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_b_ranknet"

tok_kd   = AutoTokenizer.from_pretrained(ckpt_ranknet)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_ranknet).to(device).eval()

ablation_results["RankNet"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="RankNet",
)
ablation_results["RankNet"]["params_M"] = 33.4

del model_kd, tok_kd
torch.cuda.empty_cache()

## Stage A no macro + Stage B (ADR-MSE)

In [ ]:
ckpt_adrmse = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_adrmse",
    loss_type="adr_mse",
    epochs=5, batch_size=8, lr=1e-5, alpha=0.7, patience=2, seed=42,
)

In [11]:
ckpt_adrmse = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_b_adrmse"
tok_kd   = AutoTokenizer.from_pretrained(ckpt_adrmse)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_adrmse).to(device).eval()

ablation_results["ADR-MSE"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="ADR-MSE",
)
ablation_results["ADR-MSE"]["params_M"] = 33.4

del model_kd, tok_kd
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[ADR-MSE] Warming up...
[ADR-MSE] Evaluating...

[ADR-MSE] RESULTS
Questions : 343 | Pairs: 6860 | Misses: 31
------------------------------------------------------------
Recall@5  : 0.9096
MRR@10    : 0.7827
------------------------------------------------------------
Latency   : avg=22.6ms | p50=22.6ms | p95=23.3ms
QPS       : 44.3


In [6]:
ckpt_adrmse_discount = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_discount",
    loss_type="discount",
    epochs=5, batch_size=8, lr=1e-5, alpha=0.7, patience=2, seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: discount | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.6708 | KD(discount): 0.8105 | CL: 0.3449 | Dev Acc: 0.8557


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8557)
Epoch 2/5 | Loss: 0.6244 | KD(discount): 0.7368 | CL: 0.3622 | Dev Acc: 0.8525
  No improve (1/2)
Epoch 3/5 | Loss: 0.5883 | KD(discount): 0.7019 | CL: 0.3231 | Dev Acc: 0.8492
  No improve (2/2)
Early stopping at epoch 3

discount done. Best dev acc: 0.8557


In [13]:
ckpt_adrmse_topk = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_topk",
    loss_type="topk",
    epochs=5, batch_size=8, lr=1e-5, alpha=0.7, patience=2, seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: topk | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.6618 | KD(topk): 0.7864 | CL: 0.3710 | Dev Acc: 0.8492


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8492)
Epoch 2/5 | Loss: 0.5988 | KD(topk): 0.6768 | CL: 0.4166 | Dev Acc: 0.8459
  No improve (1/2)
Epoch 3/5 | Loss: 0.5603 | KD(topk): 0.6310 | CL: 0.3955 | Dev Acc: 0.8590


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8590)
Epoch 4/5 | Loss: 0.5246 | KD(topk): 0.5756 | CL: 0.4058 | Dev Acc: 0.8525
  No improve (1/2)
Epoch 5/5 | Loss: 0.5058 | KD(topk): 0.5459 | CL: 0.4123 | Dev Acc: 0.8525
  No improve (2/2)
Early stopping at epoch 5

topk done. Best dev acc: 0.8590


In [14]:
tok_kd   = AutoTokenizer.from_pretrained(ckpt_adrmse_topk)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_adrmse_topk).to(device).eval()

ablation_results["discount"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="discount",
)
ablation_results["discount"]["params_M"] = 33.4

del model_kd, tok_kd
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[discount] Warming up...
[discount] Evaluating...

[discount] RESULTS
Questions : 343 | Pairs: 6860 | Misses: 44
------------------------------------------------------------
Recall@5  : 0.8717
MRR@10    : 0.7350
------------------------------------------------------------
Latency   : avg=22.5ms | p50=22.6ms | p95=23.4ms
QPS       : 44.4


In [ ]:
import math

def compute_extended_metrics(
    model, tokenizer, retrieve_results, questions,
    device, batch_size=32, max_len=512, warmup=10, label=""
):
    """Compute Recall@5, MRR@10, NDCG@10, MAP@10, Hit@1"""
    model.eval()

    # Warmup
    for entry in retrieve_results[:warmup]:
        pairs = [(entry["question"], c["chunk"]) for c in entry["candidates"]]
        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True,
                truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                _ = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits
    if device.type == "cuda":
        torch.cuda.synchronize()

    recall5_sum = mrr10_sum = ndcg10_sum = map10_sum = hit1_sum = 0
    total = 0

    for entry in retrieve_results:
        query = entry["question"]
        if query not in questions:
            continue

        candidates = entry["candidates"]
        gold_ids   = set(questions[query]["gold_chunk_ids"])
        pairs      = [(query, c["chunk"]) for c in candidates]
        scores     = []

        if device.type == "cuda":
            torch.cuda.synchronize()

        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True,
                truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                logits = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits.squeeze(-1)
            out = logits.detach().cpu().tolist()
            scores.extend([out] if isinstance(out, float) else out)

        if device.type == "cuda":
            torch.cuda.synchronize()

        ranked = [c for _, c in sorted(zip(scores, candidates), key=lambda x: -x[0])]

        # Binary relevance list (top-10) 
        rel = [1 if c["chunk_id"] in gold_ids else 0 for c in ranked[:10]]

        # Recall@5 
        top5 = [c["chunk_id"] for c in ranked[:5]]
        if any(g in top5 for g in gold_ids):
            recall5_sum += 1

        # Hit@1 
        if ranked[0]["chunk_id"] in gold_ids:
            hit1_sum += 1

        # MRR@10 
        for rank, c in enumerate(ranked[:10], 1):
            if c["chunk_id"] in gold_ids:
                mrr10_sum += 1.0 / rank
                break

        # NDCG@10 
        # DCG
        dcg = sum(r / math.log2(i + 2) for i, r in enumerate(rel))
        # Ideal DCG: best possible (all gold at top)
        ideal = sorted(rel, reverse=True)
        idcg  = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
        ndcg10_sum += (dcg / idcg) if idcg > 0 else 0.0

        # MAP@10 
        ap = 0.0
        hits = 0
        for i, r in enumerate(rel, 1):
            if r == 1:
                hits += 1
                ap   += hits / i
        n_gold_in_top10 = sum(rel)
        map10_sum += (ap / n_gold_in_top10) if n_gold_in_top10 > 0 else 0.0

        total += 1

    return {
        "Hit@1":     hit1_sum    / total,
        "Recall@5":  recall5_sum / total,
        "MRR@10":    mrr10_sum   / total,
        "NDCG@10":   ndcg10_sum  / total,
        "MAP@10":    map10_sum   / total,
        "total":     total,
    }

In [ ]:
import math

NEW_MODELS = {
    # "(8) No mMARCO + RankNet": ("/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_b_ranknet", 512),
    "(9) No mMARCO + ADR-MSE": ("/kaggle/working/ablation/stage_b_adrmse/best",  512),
}

for name, (ckpt, max_len) in NEW_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    tok   = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()
    n_params = round(sum(p.numel() for p in model.parameters()) / 1e6, 1)
    print(f"Loaded {n_params}M params")

    result = compute_extended_metrics(
        model=model, tokenizer=tok,
        retrieve_results=retrieve_results, questions=questions,
        device=device, batch_size=32, max_len=max_len,
        label=name,
    )
    result["params_M"] = n_params
    extended_results[name] = result

    print(f"  Hit@1    = {result['Hit@1']:.4f}")
    print(f"  Recall@5 = {result['Recall@5']:.4f}")
    print(f"  MRR@10   = {result['MRR@10']:.4f}")
    print(f"  NDCG@10  = {result['NDCG@10']:.4f}")
    print(f"  MAP@10   = {result['MAP@10']:.4f}")

    del model, tok
    torch.cuda.empty_cache()

# ── In bảng so sánh nhanh ─────────────────────────────────────────
print(f"\n{'='*80}")
print(f"{'Model':<35} {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>10} {'NDCG@10':>10}")
print("-"*80)
for name in ["(6) No mMARCO + B (KL)", "(7) No mMARCO + B (MSE)",
             "(8) No mMARCO + RankNet", "(9) No mMARCO + ADR-MSE",
             "PhoRanker", "BGE-M3 (teacher)"]:
    if name not in extended_results:
        continue
    r = extended_results[name]
    print(f"{name:<35} {r['Hit@1']:>8.4f} {r['Recall@5']:>10.4f} "
          f"{r['MRR@10']:>10.4f} {r['NDCG@10']:>10.4f}")
print("="*80)

## PhoRanker

In [ ]:
import warnings, transformers
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

tok_pho   = AutoTokenizer.from_pretrained(PHORANKER, use_fast=False)
model_pho = AutoModelForSequenceClassification.from_pretrained(PHORANKER).to(device).eval()
model_pho = model_pho.half()  # fp16
n_params  = round(sum(p.numel() for p in model_pho.parameters()) / 1e6, 1)
print(f"PhoRanker: {n_params}M params")

# PhoRanker: pre-truncate text to avoid warning spam
# Custom wrapper với max_chars để tránh warning
class TruncatedTokenizer:
    def __init__(self, tok, max_chars_q=500, max_chars_d=1000):
        self.tok     = tok
        self.max_q   = max_chars_q
        self.max_d   = max_chars_d

    def __call__(self, queries, docs, **kwargs):
        q_trunc = [q[:self.max_q] for q in queries]
        d_trunc = [d[:self.max_d] for d in docs]
        return self.tok(q_trunc, d_trunc, **kwargs)

    def __getattr__(self, name):
        return getattr(self.tok, name)

tok_pho_wrapped = TruncatedTokenizer(tok_pho)

ablation_results["PhoRanker (VI baseline)"] = benchmark_reranker(
    model=model_pho, tokenizer=tok_pho_wrapped,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=256,
    label="PhoRanker",
)
ablation_results["PhoRanker (VI baseline)"]["params_M"] = n_params

del model_pho, tok_pho
torch.cuda.empty_cache()

## Bge-reranker-v2-m3

In [ ]:
tok_bge   = AutoTokenizer.from_pretrained(BGE_DIR)
model_bge = AutoModelForSequenceClassification.from_pretrained(BGE_DIR).to(device).eval()
n_params  = round(sum(p.numel() for p in model_bge.parameters()) / 1e6, 1)
print(f"BGE-M3: {n_params}M params")

ablation_results["BGE-M3 (teacher)"] = benchmark_reranker(
    model=model_bge, tokenizer=tok_bge,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="BGE-M3",
)
ablation_results["BGE-M3 (teacher)"]["params_M"] = n_params

del model_bge, tok_bge
torch.cuda.empty_cache()

## Benchmark

In [63]:
import math

def compute_extended_metrics(
    model, tokenizer, retrieve_results, questions,
    device, batch_size=32, max_len=512, warmup=10, label=""
):
    """Compute Recall@5, MRR@10, NDCG@10, MAP@10, Hit@1"""
    model.eval()

    # Warmup
    for entry in retrieve_results[:warmup]:
        pairs = [(entry["question"], c["chunk"]) for c in entry["candidates"]]
        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True,
                truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                _ = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits
    if device.type == "cuda":
        torch.cuda.synchronize()

    recall5_sum = mrr10_sum = ndcg10_sum = map10_sum = hit1_sum = 0
    total = 0

    for entry in retrieve_results:
        query = entry["question"]
        if query not in questions:
            continue

        candidates = entry["candidates"]
        gold_ids   = set(questions[query]["gold_chunk_ids"])
        pairs      = [(query, c["chunk"]) for c in candidates]
        scores     = []

        if device.type == "cuda":
            torch.cuda.synchronize()

        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True,
                truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                logits = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits.squeeze(-1)
            out = logits.detach().cpu().tolist()
            scores.extend([out] if isinstance(out, float) else out)

        if device.type == "cuda":
            torch.cuda.synchronize()

        ranked = [c for _, c in sorted(zip(scores, candidates), key=lambda x: -x[0])]

        # Binary relevance list (top-10) 
        rel = [1 if c["chunk_id"] in gold_ids else 0 for c in ranked[:10]]

        # Recall@5 
        top5 = [c["chunk_id"] for c in ranked[:5]]
        if any(g in top5 for g in gold_ids):
            recall5_sum += 1

        # Hit@1 
        if ranked[0]["chunk_id"] in gold_ids:
            hit1_sum += 1

        # MRR@10 
        for rank, c in enumerate(ranked[:10], 1):
            if c["chunk_id"] in gold_ids:
                mrr10_sum += 1.0 / rank
                break

        # NDCG@10 
        # DCG
        dcg = sum(r / math.log2(i + 2) for i, r in enumerate(rel))
        # Ideal DCG: best possible (all gold at top)
        ideal = sorted(rel, reverse=True)
        idcg  = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
        ndcg10_sum += (dcg / idcg) if idcg > 0 else 0.0

        # MAP@10 
        ap = 0.0
        hits = 0
        for i, r in enumerate(rel, 1):
            if r == 1:
                hits += 1
                ap   += hits / i
        n_gold_in_top10 = sum(rel)
        map10_sum += (ap / n_gold_in_top10) if n_gold_in_top10 > 0 else 0.0

        total += 1

    return {
        "Hit@1":     hit1_sum    / total,
        "Recall@5":  recall5_sum / total,
        "MRR@10":    mrr10_sum   / total,
        "NDCG@10":   ndcg10_sum  / total,
        "MAP@10":    map10_sum   / total,
        "total":     total,
    }

In [ ]:
import json

INPUT  = "/kaggle/working/domain_train_with_teacher_scores.jsonl"
OUTPUT = "/kaggle/working/domain_train_margin_filtered.jsonl"

kept = removed_neg_margin = removed_giveaway = 0

with open(INPUT) as fin, open(OUTPUT, "w") as fout:
    for line in fin:
        d = json.loads(line)
        
        # Filter 1: Teacher margin phải dương
        if d["teacher_margin"] <= 0.1:
            removed_neg_margin += 1
            continue
        
        # Filter 2: Negative không chứa query (giveaway check)
        query_lower = d["query"].lower().strip()
        neg_lower   = d["negative"].lower()
        
        # Nếu negative chứa >50% từ của query → giveaway
        query_words = set(query_lower.split())
        neg_words   = set(neg_lower.split())
        overlap     = len(query_words & neg_words) / max(len(query_words), 1)
        
        if overlap > 0.7:  # giveaway threshold
            removed_giveaway += 1
            continue
        
        # Filter 3: Negative không quá giống positive (literal text match)
        if d["negative"][:200] == d["positive"][:200]:
            removed_giveaway += 1
            continue
        
        fout.write(line)
        kept += 1

print(f"Kept: {kept}")
print(f"Removed (neg margin):  {removed_neg_margin}")
print(f"Removed (giveaway):    {removed_giveaway}")
print(f"Total removed: {removed_neg_margin + removed_giveaway}")

In [ ]:
import numpy as np

margins = []
with open(OUTPUT) as f:
    for line in f:
        d = json.loads(line)
        margins.append(d["teacher_margin"])

margins = np.array(margins)
print(f"\nAfter filter:")
print(f"Total: {len(margins)}")
print(f"Mean margin: {margins.mean():.4f}")
print(f"Min margin:  {margins.min():.4f}")
print(f"Max margin:  {margins.max():.4f}")
print(f"Negative margins: {(margins < 0).sum()}")

In [ ]:
# ── MarginMSE Dataset ────────────────────────────────────────────
class MarginMSEDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=512):
        self.data = []
        with open(path) as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tok     = tokenizer
        self.max_len = max_length

    def encode(self, query, passage):
        return self.tok(
            query, passage,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        d   = self.data[idx]
        pos = self.encode(d["query"], d["positive"])
        neg = self.encode(d["query"], d["negative"])
        return {
            "pos_input_ids":      pos["input_ids"].squeeze(),
            "pos_attention_mask": pos["attention_mask"].squeeze(),
            "neg_input_ids":      neg["input_ids"].squeeze(),
            "neg_attention_mask": neg["attention_mask"].squeeze(),
            "teacher_margin":     torch.tensor(d["teacher_margin"], dtype=torch.float),
        }

    def __len__(self):
        return len(self.data)


# ── Margin-MSE Loss ──────────────────────────────────────────────
class MarginMSELoss(nn.Module):
    """Hofstätter et al. 2020 — student margin matches teacher margin"""
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, student_pos, student_neg, teacher_margin):
        student_margin = student_pos - student_neg
        return self.mse(student_margin, teacher_margin)


# ── Train function với Margin-MSE ────────────────────────────────
def train_margin_mse(
    base_checkpoint,
    train_path,
    dev_path,
    output_dir,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=512,
    patience=2,
    seed=42,
):
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
    model     = AutoModelForSequenceClassification.from_pretrained(
        base_checkpoint, num_labels=1, ignore_mismatched_sizes=True
    )
    device = torch.device("cuda")
    model.to(device)
    print(f"Device: {device} | Seed: {seed}")

    train_dataset = MarginMSEDataset(train_path, tokenizer, max_length)
    dev_dataset   = PairwiseDataset([dev_path], tokenizer, max_length)
    print(f"Train: {len(train_dataset):,} | Dev: {len(dev_dataset):,}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=0,
                              worker_init_fn=lambda w: set_seed(seed + w))
    dev_loader   = DataLoader(dev_dataset, batch_size=batch_size, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    criterion = MarginMSELoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc   = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            pos_logits = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device)
            ).logits.squeeze(-1)
            neg_logits = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device)
            ).logits.squeeze(-1)

            loss = criterion(pos_logits, neg_logits,
                             batch["teacher_margin"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        acc = evaluate(model, dev_loader, device)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Domain Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tokenizer.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nBest acc: {best_acc:.4f}")
    return f"{output_dir}/best"

In [ ]:
import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

In [ ]:
seed_results_mse = {}

for seed in [42, 0, 123]:
    set_seed(seed)
    ckpt = train_margin_mse(
        base_checkpoint="/kaggle/input/datasets/tranquanghuy2809/tqhtqh/minilm_stage_a",
        train_path="/kaggle/working/domain_train_with_teacher_scores.jsonl",
        dev_path="/kaggle/input/datasets/tranquanghuy2809/tqhtqh/domain_train_final_dev.jsonl",
        output_dir=f"/kaggle/working/ckpt_mse_seed{seed}",
        epochs=5, batch_size=16, lr=1e-5, patience=2, seed=seed,
    )
    device     = torch.device("cuda")
    tok_eval   = AutoTokenizer.from_pretrained(ckpt)
    model_eval = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()
    result = benchmark_reranker(
        model=model_eval, tokenizer=tok_eval,
        retrieve_results=retrieve_results, questions=questions,
        device=device, max_questions=len(retrieve_results),
        batch_size=32, max_len=512, label=f"Margin-MSE seed={seed}"
    )
    seed_results_mse[seed] = result
    del model_eval, tok_eval
    torch.cuda.empty_cache()
    print(f"Seed {seed} → Recall@5={result['Recall@5']:.4f} | MRR@10={result['MRR@10']:.4f}")

print(f"\nMRR@10:   {np.mean([r['MRR@10'] for r in seed_results_mse.values()]):.4f} ± {np.std([r['MRR@10'] for r in seed_results_mse.values()]):.4f}")
print(f"Recall@5: {np.mean([r['Recall@5'] for r in seed_results_mse.values()]):.4f} ± {np.std([r['Recall@5'] for r in seed_results_mse.values()]):.4f}")